# Targeted Control — V=32, k=3, L=3
## Training-budget and capacity scaling

This notebook addresses the reviewer-style question raised by the Part 1 audit:

> Why does the information gap vanish at L=k while a large model/optimization gap remains for V=32, k=3?

It reuses the existing persistent project folder:

`Markov_Context_Paper/Part1/`

and runs only a focused control at:

- V = 32
- k = 3
- L = 3
- seeds = 0,1,2
- training budgets = 12k, 24k, 48k steps
- base and larger Transformer configurations

All runs are checkpointed, resumable, and stored separately.


In [ ]:

import os, sys, json, time, random, hashlib, zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("AMP:", "BF16" if USE_BF16 else ("FP16" if torch.cuda.is_available() else "OFF"))


In [ ]:

PROJECT_ROOT = Path.cwd() / "Markov_Context_Paper" / "Part1"
RUNS_DIR = PROJECT_ROOT / "runs"
EXPORTS_DIR = PROJECT_ROOT / "exports"
CSV_DIR = EXPORTS_DIR / "csv"
THEORY_DIR = PROJECT_ROOT / "theory_cache"

for p in [PROJECT_ROOT, RUNS_DIR, EXPORTS_DIR, CSV_DIR, THEORY_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)


In [ ]:

SEEDS = [0,1,2]

BASE_BATCH = 1024
BASE_LR = 1e-3
WEIGHT_DECAY = 1e-2
SAVE_EVERY = 500
EVAL_BATCH = 4096
CHAIN_WARMUP = 64

THEORY_SAMPLES = 500_000
THEORY_BATCH = 4096

# Focused scaling grid
BUDGETS = [12_000, 24_000, 48_000]

MODEL_CONFIGS = {
    "base": {
        "d_model": 256,
        "n_layers": 3,
        "n_heads": 8,
        "dim_ff": 512,
    },
    "large": {
        "d_model": 512,
        "n_layers": 4,
        "n_heads": 8,
        "dim_ff": 1024,
    },
}

print("Seeds:", SEEDS)
print("Budgets:", BUDGETS)
print("Models:", MODEL_CONFIGS)


In [ ]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state().cpu(),
    }
    if torch.cuda.is_available():
        state["torch_cuda"] = [x.cpu() for x in torch.cuda.get_rng_state_all()]
    return state

def set_rng_state(state):
    if state is None:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])

    cpu_state = state["torch_cpu"]
    if isinstance(cpu_state, torch.Tensor):
        cpu_state = cpu_state.detach().cpu().to(torch.uint8)
    torch.set_rng_state(cpu_state)

    if torch.cuda.is_available() and "torch_cuda" in state:
        cuda_states = []
        for s in state["torch_cuda"]:
            if isinstance(s, torch.Tensor):
                s = s.detach().cpu().to(torch.uint8)
            cuda_states.append(s)
        torch.cuda.set_rng_state_all(cuda_states)

def safe_json(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=safe_json)

def stable_hash(d, n=12):
    payload = json.dumps(d, sort_keys=True, default=safe_json).encode()
    return hashlib.sha256(payload).hexdigest()[:n]


In [ ]:

def context_to_index(context, V):
    idx = torch.zeros(context.shape[:-1], dtype=torch.long, device=context.device)
    for j in range(context.shape[-1]):
        idx = idx * V + context[..., j]
    return idx

def generate_kernel(V, k, seed, sparse_s=None, device=DEVICE):
    g = torch.Generator(device=device)
    g.manual_seed(seed + 10_000_019)
    logits = torch.randn((V**k, V), generator=g, device=device, dtype=torch.float32)
    return torch.softmax(logits, dim=-1)

class PersistentMarkovSampler:
    def __init__(self, P, V, k, batch_size, max_history, seed, warmup_steps=64):
        self.P, self.V, self.k = P, V, k
        self.batch_size = batch_size
        self.max_history = max(max_history, k)
        self.gen = torch.Generator(device=DEVICE)
        self.gen.manual_seed(seed + 4_000_003)
        self.history = torch.randint(
            0, V, (batch_size, self.max_history),
            generator=self.gen, device=DEVICE
        )
        for _ in range(warmup_steps):
            self.advance()

    @torch.no_grad()
    def draw_next(self):
        idx = context_to_index(self.history[:, -self.k:], self.V)
        probs = self.P[idx]
        return torch.multinomial(probs, 1, generator=self.gen).squeeze(1)

    @torch.no_grad()
    def advance(self):
        y = self.draw_next()
        self.history = torch.cat([self.history[:, 1:], y[:, None]], dim=1)

    @torch.no_grad()
    def sample(self, L):
        x = self.history[:, -L:].clone()
        y = self.draw_next()
        self.history = torch.cat([self.history[:, 1:], y[:, None]], dim=1)
        return x, y


In [ ]:

@torch.no_grad()
def estimate_theory(P, V, k, max_L, n_samples=THEORY_SAMPLES, batch_size=THEORY_BATCH, seed=12345):
    max_history = max(max_L, k)
    sampler = PersistentMarkovSampler(P, V, k, batch_size, max_history, seed, 256)

    hstar_sum = 0.0
    seen = 0

    while seen < n_samples:
        b = min(batch_size, n_samples - seen)

        if b != batch_size:
            hist = PersistentMarkovSampler(
                P, V, k, b, max_history, seed + seen + 1, 256
            ).history
        else:
            hist = sampler.history

        p_full = P[context_to_index(hist[:, -k:], V)].double()
        hstar_sum += (-(p_full * torch.log(p_full.clamp_min(1e-30))).sum(-1)).sum().item()

        if b == batch_size:
            sampler.advance()
        seen += b

    H_star = hstar_sum / seen

    out = {}
    for L in range(1, max_L + 1):
        # We only need L=k=3 here; by Markov sufficiency H_L=H*
        H_L = H_star if L >= k else None
        out[L] = {
            "H_star": float(H_star),
            "H_L": None if H_L is None else float(H_L),
            "info_gap": None if H_L is None else float(H_L - H_star),
        }
    return out

def theory_path(V, k, seed, max_L):
    return THEORY_DIR / f"target_V{V}_k{k}_seed{seed}_L{max_L}_N{THEORY_SAMPLES}.json"

def get_theory(V, k, seed, max_L=3, force=False):
    p = theory_path(V, k, seed, max_L)

    if p.exists() and not force:
        raw = json.load(open(p, "r", encoding="utf-8"))
        return {int(a): b for a, b in raw.items()}

    P = generate_kernel(V, k, seed)
    out = estimate_theory(
        P, V, k, max_L,
        n_samples=THEORY_SAMPLES,
        batch_size=THEORY_BATCH,
        seed=seed + 1234
    )

    save_json(p, out)
    return out


In [ ]:

class TinyCausalTransformer(nn.Module):
    def __init__(self, V, max_context, d_model=256, n_layers=3, n_heads=8, dim_ff=512):
        super().__init__()

        self.token_emb = nn.Embedding(V, d_model)
        self.pos_emb = nn.Embedding(max_context, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_ff,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, V)

    def forward(self, x):
        _, L = x.shape
        pos = torch.arange(L, device=x.device)[None, :]
        h = self.token_emb(x) + self.pos_emb(pos)

        mask = torch.triu(
            torch.ones((L, L), dtype=torch.bool, device=x.device),
            diagonal=1
        )

        h = self.encoder(h, mask=mask)
        return self.head(self.norm(h[:, -1, :]))

def nparams(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:

@dataclass
class RunConfig:
    experiment: str
    V: int
    k: int
    L: int
    seed: int
    steps: int
    batch_size: int
    lr: float
    weight_decay: float
    d_model: int
    n_layers: int
    n_heads: int
    dim_ff: int
    eval_batch: int
    save_every: int

    def run_id(self):
        d = asdict(self)
        return (
            f"{self.experiment}_V{self.V}_k{self.k}_L{self.L}"
            f"_seed{self.seed}_steps{self.steps}_d{self.d_model}"
            f"_ly{self.n_layers}_{stable_hash(d)}"
        )

def rdir(cfg): return RUNS_DIR / cfg.experiment / cfg.run_id()
def p_done(cfg): return rdir(cfg) / "DONE"
def p_ckpt(cfg): return rdir(cfg) / "checkpoint_last.pt"
def p_best(cfg): return rdir(cfg) / "checkpoint_best.pt"
def p_kernel(cfg): return rdir(cfg) / "kernel.pt"
def p_hist(cfg): return rdir(cfg) / "metrics.csv"
def p_final(cfg): return rdir(cfg) / "final_metrics.json"


In [ ]:

@torch.no_grad()
def evaluate(model, P, V, k, L, batch_size, seed):
    model.eval()

    sampler = PersistentMarkovSampler(
        P, V, k, batch_size,
        max(L, k),
        seed,
        128
    )

    x, y = sampler.sample(L)

    with torch.autocast(
        "cuda",
        dtype=AMP_DTYPE,
        enabled=torch.cuda.is_available()
    ):
        logits = model(x)

    return float(F.cross_entropy(logits.float(), y).item())

def save_ckpt(path, model, optimizer, scheduler, step, best_eval, history, cfg):
    payload = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "step": step,
        "best_eval": best_eval,
        "history": history,
        "rng_state": get_rng_state(),
        "config": asdict(cfg),
    }

    tmp = Path(str(path) + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)

def train_run(cfg, H_star=None, H_L=None, force=False):
    rd = rdir(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    save_json(rd / "config.json", asdict(cfg))

    if p_done(cfg).exists() and not force:
        print("SKIP DONE:", cfg.run_id())
        return json.load(open(p_final(cfg), "r", encoding="utf-8"))

    set_seed(cfg.seed)

    if p_kernel(cfg).exists():
        P = torch.load(p_kernel(cfg), map_location=DEVICE)
    else:
        P = generate_kernel(cfg.V, cfg.k, cfg.seed).detach().cpu()
        torch.save(P, p_kernel(cfg))
        P = P.to(DEVICE)

    model = TinyCausalTransformer(
        cfg.V,
        max_context=max(cfg.L, cfg.k + 3),
        d_model=cfg.d_model,
        n_layers=cfg.n_layers,
        n_heads=cfg.n_heads,
        dim_ff=cfg.dim_ff,
    ).to(DEVICE)

    optimizer = AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=cfg.steps
    )

    start = 0
    best_eval = float("inf")
    history = []

    if p_ckpt(cfg).exists() and not force:
        ck = torch.load(p_ckpt(cfg), map_location=DEVICE)

        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"])

        start = int(ck["step"]) + 1
        best_eval = float(ck["best_eval"])
        history = ck.get("history", [])

        set_rng_state(ck.get("rng_state"))

        print("RESUME:", cfg.run_id(), "from", start)

    sampler = PersistentMarkovSampler(
        P,
        cfg.V,
        cfg.k,
        cfg.batch_size,
        max(cfg.L, cfg.k),
        cfg.seed + 888001,
        CHAIN_WARMUP
    )

    model.train()
    t0 = time.time()

    for step in range(start, cfg.steps):
        x, y = sampler.sample(cfg.L)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            "cuda",
            dtype=AMP_DTYPE,
            enabled=torch.cuda.is_available()
        ):
            logits = model(x)
            loss = F.cross_entropy(logits.float(), y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if step == 0 or (step + 1) % 100 == 0:
            history.append({
                "step": step + 1,
                "train_loss": float(loss.item()),
                "lr": float(scheduler.get_last_lr()[0]),
                "elapsed_s": time.time() - t0,
            })

        if (step + 1) % cfg.save_every == 0 or step + 1 == cfg.steps:
            ev = evaluate(
                model, P,
                cfg.V, cfg.k, cfg.L,
                cfg.eval_batch,
                cfg.seed + 77777 + step
            )

            history.append({
                "step": step + 1,
                "eval_loss": ev,
                "lr": float(scheduler.get_last_lr()[0]),
                "elapsed_s": time.time() - t0,
            })

            pd.DataFrame(history).to_csv(p_hist(cfg), index=False)

            if ev < best_eval:
                best_eval = ev
                save_ckpt(
                    p_best(cfg),
                    model, optimizer, scheduler,
                    step, best_eval, history, cfg
                )

            save_ckpt(
                p_ckpt(cfg),
                model, optimizer, scheduler,
                step, best_eval, history, cfg
            )

            print(
                f"{cfg.experiment} "
                f"V={cfg.V} k={cfg.k} L={cfg.L} "
                f"seed={cfg.seed} steps={cfg.steps} "
                f"d={cfg.d_model} "
                f"step={step+1}/{cfg.steps} "
                f"train={loss.item():.4f} eval={ev:.4f}"
            )

    if p_best(cfg).exists():
        best = torch.load(p_best(cfg), map_location=DEVICE)
        model.load_state_dict(best["model"])

    test_loss = evaluate(
        model, P,
        cfg.V, cfg.k, cfg.L,
        cfg.eval_batch,
        cfg.seed + 999999
    )

    result = {
        **asdict(cfg),
        "run_id": cfg.run_id(),
        "run_dir": str(rd),
        "n_params": nparams(model),
        "test_loss": test_loss,
        "H_star": H_star,
        "H_L": H_L,
        "total_excess": None if H_star is None else test_loss - H_star,
        "info_gap": None if H_star is None or H_L is None else H_L - H_star,
        "model_gap": None if H_L is None else test_loss - H_L,
        "best_eval": best_eval,
        "elapsed_s": time.time() - t0,
    }

    save_json(p_final(cfg), result)
    p_done(cfg).write_text("completed\n", encoding="utf-8")

    return result


## Run targeted V32-k3-L3 scaling control

In [ ]:

START = time.time()
results = []

V = 32
k = 3
L = 3

for seed in SEEDS:
    theory = get_theory(V, k, seed, max_L=L)

    for model_name, mc in MODEL_CONFIGS.items():
        for steps in BUDGETS:

            cfg = RunConfig(
                experiment=f"v32k3_target_{model_name}",
                V=V,
                k=k,
                L=L,
                seed=seed,
                steps=steps,
                batch_size=BASE_BATCH,
                lr=BASE_LR,
                weight_decay=WEIGHT_DECAY,
                d_model=mc["d_model"],
                n_layers=mc["n_layers"],
                n_heads=mc["n_heads"],
                dim_ff=mc["dim_ff"],
                eval_batch=EVAL_BATCH,
                save_every=SAVE_EVERY,
            )

            res = train_run(
                cfg,
                H_star=theory[L]["H_star"],
                H_L=theory[L]["H_L"],
            )

            res["model_name"] = model_name
            results.append(res)

df = pd.DataFrame(results)
df.to_csv(
    CSV_DIR / "v32k3_targeted_scaling_raw.csv",
    index=False
)

print("Targeted scaling complete.")


In [ ]:

# Aggregate results
target = pd.DataFrame(results)

summary = (
    target
    .groupby(["model_name","steps"], as_index=False)
    .agg(
        test_loss_mean=("test_loss","mean"),
        test_loss_std=("test_loss","std"),
        H_star_mean=("H_star","mean"),
        model_gap_mean=("model_gap","mean"),
        model_gap_std=("model_gap","std"),
        n=("seed","count"),
    )
)

summary.to_csv(
    CSV_DIR / "v32k3_targeted_scaling_summary.csv",
    index=False
)

display(summary)

output_zip = Path.cwd() / "V32K3_Targeted_Control_RESULTS.zip"

allowed = {".csv",".json",".png",".pdf",".txt"}

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for file in PROJECT_ROOT.rglob("*"):
        if file.is_file() and file.suffix.lower() in allowed:
            z.write(file, file.relative_to(PROJECT_ROOT))

print("Elapsed hours:", round((time.time()-START)/3600, 2))
print("ZIP:", output_zip)
print("ZIP size MB:", round(output_zip.stat().st_size/1024**2, 2))
